# PDF Question-Answering with LangChain + Gemini

A Retrieval-Augmented Generation (RAG) pipeline that answers questions using **only** the
content of a PDF. If the answer isn't in the document, it says so instead of guessing.

**Why RAG:** Gemini has never seen this PDF, and pasting all 82 pages into the prompt is
both too large and less accurate (models lose track of detail in long context). Instead we
find the few passages relevant to each question and send only those.

**Pipeline:**

| Step | Component | Purpose |
|---|---|---|
| 1. Load | `PyPDFLoader` | PDF → one Document per page |
| 2. Split | `RecursiveCharacterTextSplitter` | Pages → overlapping chunks |
| 3. Embed | `GoogleGenerativeAIEmbeddings` | Chunks → vectors (meaning as numbers) |
| 4. Store | `FAISS` | Vector index for fast similarity search |
| 5. Retrieve | `as_retriever(...)` | Question → most relevant chunks |
| 6. Generate | `ChatGoogleGenerativeAI` + prompt | Chunks + question → grounded answer |

## 0. Setup

Install dependencies. Note that pip names and import names differ: install `faiss-cpu`,
import `faiss`.

`pypdf` is needed by `PyPDFLoader` — the import succeeds without it, but loading a PDF fails.

Restart the kernel after installing, or new packages won't be importable.

In [81]:
%pip install langchain_community
%pip install faiss-cpu

Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   - -------------------------------------- 0.8/16.3 MB 2.9 MB/s eta 0:00:06
   ---- ----------------------------------- 1.8/16.3 MB 3.8 MB/s eta 0:00:04
   ------- -------------------------------- 3.1/16.3 MB 4.7 MB/s eta 0:00:03
   ----------- ---------------------------- 4.7/16.3 MB 5.6 MB/s eta 0:00:03
   --------------- ------------------------ 6.3/16.3 MB 5.7 MB/s eta 0:00:02
   ------------------- -------------------- 8.1/16.3 MB 6.2 MB/s eta 0:00:02
   ---------------------- ----------------- 9.2/16.3 MB 6.0 MB/s eta 0:00:02
   ------------------------- -------------- 10.2/16.3 MB 5.9 MB/s eta 0:00:02
   ------------------------------ --------- 12.3/16.3 MB 6.3 MB/s eta 0:00:01
   ---------------------------------- ----- 13.9/16.3 MB 6.4 MB/s eta 0:00:01
   ------------------

In [1]:
%pip install pypdf

Note: you may need to restart the kernel to use updated packages.


## 1. Imports

LangChain is split across packages, each providing different pieces:

- `langchain_text_splitters` → chunking
- `langchain_community.document_loaders` → `PyPDFLoader`
- `langchain_google_genai` → Gemini chat + embeddings

In [58]:
from langchain_text_splitters import CharacterTextSplitter,RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

In [59]:
import os
import getpass
from langchain_google_genai import ChatGoogleGenerativeAI

## 2. API key

The key is read from the environment variable `GOOGLE_API_KEY`, which is the name
`langchain-google-genai` looks for — so it never has to be passed explicitly.

`getpass` is used instead of a literal string so the key is never saved inside the
notebook file (which would leak it to git). Entered fresh each session.

To re-enter a mistyped key, restart the kernel — the `if not` check skips the prompt
once the variable is set.

In [60]:
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass('Enter your Gemini API key:')
else:
    print('GOOGLE API KEY IS ALREADY STORED IN ENV.')

GOOGLE API KEY IS ALREADY STORED IN ENV.


## 3. Test the Gemini connection

A smoke test before building anything: confirms the key is valid and the model name exists.

`.text` returns plain text. 

**Note:** `gemini` models use fixed sampling, so `temperature=0` is ignored. All
grounding must come from the prompt, not from sampling settings.

In [61]:

gemini_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

llm_res1 = gemini_llm.invoke("What is LangChain?")
print(llm_res1.text)

**LangChain** is an open-source development framework designed to simplify the creation of applications powered by Large Language Models (LLMs), such as OpenAI’s GPT-4, Google's Gemini, and open-source models like Llama 3. 

Think of LangChain as the "glue" or an operating system for LLMs. While an LLM is like a brilliant brain trapped in a box (it knows a lot, but has no memory of past conversations, cannot access the real world, and can't use tools), LangChain provides the arms, legs, and senses to connect that brain to the outside world.

Here is a breakdown of why LangChain exists and what it does:

### 1. The Core Problem LangChain Solves
If you want to build an AI chatbot that can answer questions based on your company's private PDF documents, you can't just send the whole PDF to ChatGPT (it won't fit, and it's too expensive). You need a system that:
* Reads and breaks down the PDF.
* Searches through it to find relevant paragraphs.
* Feeds only those paragraphs to the LLM.
* Rem

## 4. Load the PDF

`PyPDFLoader` returns a **list of `Document` objects, one per page**, each with:

- `.page_content` — extracted text
- `.metadata` — source file and page number (used later to cite answers)

**Check the output is real text.** Minor spacing artifacts ("gainin g") are normal and harmless;
if they're everywhere, `PyMuPDFLoader` often extracts more cleanly.

In [62]:
pdf_loader = PyPDFLoader('PYTHON NOTES A-Z 8.pdf')
pages = pdf_loader.load()
print("Pages loaded:", len(pages))

for p in pages[:5]:          
    print("PAGE CONTENT:", p.page_content)
    print("METADATA:", p.metadata)
    print("-" * 50)
    

Pages loaded: 82
PAGE CONTENT: Python Introduction Notes  Made By: Ishan Singh Page 1 
 
Introduction to Python 
Introduction 
Python is a programming language that combines the features of C and Java. It 
offers elegant style of developing programs like C. when the programmers want 
to go for object orientation, Python offers classes and objects like Java. 
Coming to a bit of history, Python was developed by Guido Van Rossum in the 
year 1991 at the Center for Mathematics and Computer Science managed by 
the Dutch Government. 
Van Rossum picked the name Python for the new language from the TV show, 
Monty Python’s Flying Circus. Python’s first working version was ready by early 
1990 and Van Rossum released it for a public on February 20, 1991. The logo of 
Python shows two intertwined snakes as shown below: 
 
 
Features of Python 
There are various reasons why Python is gainin g good  popularity in the 
programming community. The following are some of t he important features of 
Pyt

## 5. Split into chunks

Pages are too big to embed as single units, so they're cut into smaller pieces.

**Why `Recursive`CharacterTextSplitter?** It tries separators in order —
paragraph (`\n\n`) → line (`\n`) → word (` `) → character — falling back only when a
piece is still too big. Plain `CharacterTextSplitter` splits on one separator only and
leaves oversized chunks when a paragraph exceeds `chunk_size`.

**Why `chunk_overlap=150`?** The recursive splitter decides *where* to cut; overlap
handles the fact that a cut exists at all. Consider two cleanly split paragraphs:

    chunk 0: "...Python was created by Guido van Rossum."
    chunk 1: "He released the first version in 1991."

Chunk 1 holds the date but says "He" — so it matches a question naming Guido poorly, and
may never be retrieved. Overlap repeats chunk 0's tail at the start of chunk 1, keeping
each chunk **self-contained**. Cost: ~15% more chunks. Rule of thumb: 10–20% of chunk size.

**Why `.split_documents()` and not `.split_text()`?** `split_documents` takes Document
objects and **preserves metadata**, so every chunk still knows its page number.
`split_text` takes a plain string and returns bare strings — page numbers lost.

**Why 1400?** Originally 1000 (→ 134 chunks), raised to 1400 (→ 97 chunks) to fit under
Gemini's free-tier limit of 100 embedding requests per minute. Bigger chunks carry more
context but produce blurrier embeddings that match many questions weakly instead of one
strongly.

In [63]:
rec_char_splitter = RecursiveCharacterTextSplitter(chunk_size=1400, chunk_overlap = 150)
chunks = rec_char_splitter.split_documents(pages)


### Verify the overlap

The tail of chunk 0 should reappear at the head of chunk 1, and `metadata` should still
carry `source` and `page_label`. Slice the list when printing — 97 full chunks floods
the notebook.

In [64]:
print('total code chunks:  ',len(chunks))
for chunk in chunks[:5]:
    print(chunk)
    print('-'*100)

total code chunks:   97
page_content='Python Introduction Notes  Made By: Ishan Singh Page 1 
 
Introduction to Python 
Introduction 
Python is a programming language that combines the features of C and Java. It 
offers elegant style of developing programs like C. when the programmers want 
to go for object orientation, Python offers classes and objects like Java. 
Coming to a bit of history, Python was developed by Guido Van Rossum in the 
year 1991 at the Center for Mathematics and Computer Science managed by 
the Dutch Government. 
Van Rossum picked the name Python for the new language from the TV show, 
Monty Python’s Flying Circus. Python’s first working version was ready by early 
1990 and Van Rossum released it for a public on February 20, 1991. The logo of 
Python shows two intertwined snakes as shown below: 
 
 
Features of Python 
There are various reasons why Python is gainin g good  popularity in the 
programming community. The following are some of t he important features 

## 6. Embeddings and the vector store

**Embeddings** convert text into vectors — lists of numbers encoding meaning. Texts with
similar meaning land near each other, which is why "What's the refund window?" can match
"Returns are accepted within 30 days" with no shared keywords. 

**FAISS** (Meta's similarity-search library) stores the vectors and finds the nearest ones
to a query. At 97 chunks brute force would do; FAISS is here because it's the standard
component, it scales, and it handles persistence.

**Critical rule:** the index and the queries must use the **same embedding model**.
Different models produce different vector spaces — mixing them gives a dimension error, or
worse, results that look plausible but are meaningless. Create the embeddings object once,
in a variable, and pass that variable everywhere.

**Free-tier limits** (these drove several design choices):

| Limit | Value | Applies to |
|---|---|---|
| Embeddings | 100 / minute | every chunk at index time, plus 1 per query |
| Chat | ~20 / day | each `rag_chain.invoke()` |

Quotas are **per model**, but all Gemini embedding models share the 100/min cap, so
switching models doesn't avoid it — fewer chunks or batching does.


In [65]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

In [66]:
faiss_vectorstore = FAISS.from_documents(documents=chunks,embedding=GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001'))

### Test retrieval before spending LLM calls

Retrieval only touches the local FAISS index (plus one embedding call for the query), so
it's effectively free — while generation is capped at ~20/day.

**Debug in this order:** if retrieval returns the wrong chunks, no prompt can fix it. Verify
the right page ranks first, *then* move on to generation.


In [67]:

results = faiss_vectorstore.similarity_search("who created Python?", k=3)
for r in results:
    print(r.metadata["page_label"], "|", r.page_content[:200])
    print("-" * 60)

1 | Python Introduction Notes  Made By: Ishan Singh Page 1 
 
Introduction to Python 
Introduction 
Python is a programming language that combines the features of C and Java. It 
offers elegant style of d
------------------------------------------------------------
2 | Python Introduction Notes  Made By: Ishan Singh Page 2 
 
 Open Source: There is no need to pay for Python software . Python can 
be freely downloaded from www.python.org website. 
 High level langu
------------------------------------------------------------
3 | Python Introduction Notes  Made By: Ishan Singh Page 3 
 
Python Virtual Machine (PVM) 
The role of Python Virtual  Machine ( PVM) is to convert the byte c ode 
instructions into machine code so that 
------------------------------------------------------------


## 7. The grounded prompt

This is the heart of the project. The "answer only from the PDF" behavior isn't a LangChain
feature — it's prompt engineering.

Design decisions in the wording:

- **"the context below", not "the PDF".** The model never sees a PDF, only the few chunks
  pasted into `{context}`. Saying "the PDF" invites it to assume the document covers things
  it can't see, and fill gaps from memory.
- **The refusal sentence is quoted exactly.** Without a specific sentence the model
  improvises — sometimes "I don't know", sometimes a hedge, sometimes an answer from
  training. Specifying the string makes the output predictable.
- **"Do not guess or infer" is separate from "no outside knowledge".** Different failure
  modes: a model can stay inside the context and still invent a link between two chunks.
- **Ending with `Answer:`** cues the model that its turn starts there, suppressing preamble
  like "Based on the provided context...".

`{context}` and `{question}` are template variables filled at runtime — names are
case-sensitive and must match the keys passed in, or you get a `KeyError`.


In [68]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""You answer questions about a specific document, using only the text provided to you below.

Rules:
- Use only the information in the context below to answer the question.
- Do not use any outside knowledge, even if you know the answer.
- If the context does not contain the answer, reply exactly: "Sorry, this information is not available in given pdf"
- Do not guess, infer beyond the context, or fill in gaps from memory.

Context:
{context}

Question:
{question}

Answer:""")

## 8. Output parser

`StrOutputParser` extracts plain text from the model's reply. Without it you get an
`AIMessage` — and on newer Gemini models, `.content` is a *list of content blocks* rather
than a string.



In [69]:
from langchain_core.output_parsers import StrOutputParser
string_par = StrOutputParser()

## 9. Retrievers

A **retriever** is a standard interface — query in, documents out — so chains can be
written against it regardless of what's behind it (vector store, web search, SQL). That's
why the chain uses a retriever rather than calling `similarity_search` directly.

| | Behavior |
|---|---|
| `retriever1` similarity | Returns exactly `k` nearest chunks, always |
| `retriever2` MMR | Fetches `fetch_k=10` candidates, picks `k=3` that are relevant **and** different from each other — useful when overlapping chunks make the top results near-duplicates |
| `retriever3` score threshold | Drops anything scoring below `0.5`, so it returns **0 to k** documents |

`retriever3` is the one used in the chain, and the reason is the out-of-scope requirement.
The first two always return 3 chunks — ask "what is the capital of France?" and the model
receives three plausible-looking Python passages and has to *decide* they're irrelevant.
With a threshold, the context arrives **empty**, so the refusal is forced by the data rather
than trusted to the model's judgment.

`k` = how many chunks to show the model, not how many answers to produce. All retrieved
chunks are joined into one `{context}` string and produce a single answer.

In [70]:
retriever1 = faiss_vectorstore.as_retriever(search_kwargs={"k": 3})

retriever2 = faiss_vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 3, "fetch_k": 10})

retriever3 = faiss_vectorstore.as_retriever(search_type="similarity_score_threshold", search_kwargs={"k": 3, "score_threshold": 0.5})

### Documents → string

The retriever returns `Document` objects; `{context}` needs text. This joins their
`page_content` with blank lines.

`docs` is a **parameter** — it receives whatever the retriever produced for *this*
question, not the global `chunks` list. 

In [71]:
def format_docs(docs):       
    return "\n\n".join(d.page_content for d in docs)

### Compare retrievers (free — no LLM involved)

All three retrieve the correct page for the in-scope question, so nothing is lost by using
the threshold — and only `thresh` returns nothing for the out-of-scope one. Threshold 0.5
landed in the gap on the first try; no tuning needed.

for name, r in [("sim", retriever1), ("mmr", retriever2), ("thresh", retriever3)]:
    for q in ["Who created Python?", "What is the capital of France?"]:
        docs = r.invoke(q)
        pages = [d.metadata["page_label"] for d in docs]
        print(f"{name:7} | {q:35} | {len(docs)} docs | pages {pages}")
    print()

## 10. Assemble the RAG chain

LCEL's `|` pipes each component's output into the next, building a `RunnableSequence`.

The dict runs **two branches on the same input**: the question goes to the retriever
(→ relevant text) *and*, via `RunnablePassthrough()`, straight through unchanged
(→ the question itself). The prompt needs both.

`RunnablePassthrough` is only needed when building a dict where one value is the untouched
input. A purely linear chain (`prompt | llm | parser | prompt2 | llm`) doesn't need it —
each step's single output feeds the next step's single variable.

In [74]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = ({"context": retriever3 | format_docs, "question": RunnablePassthrough()} | prompt | gemini_llm | string_par)

## 11. Testing

Each `invoke` costs 1 embedding call + 1 chat call, and chat is capped at ~20/day.

| Test | Expected | Tests |
|---|---|---|
| "Who created Python?" | Answer from page 1 | The happy path |
| "What is the capital of France?" | Refusal | The **retriever** (0 docs) |
| "What is a Python decorator?" | Refusal | The **prompt** — a Python topic absent from the notes |

The decorator case is the meaningful one: it's semantically close to the document, so
chunks may well be retrieved that look relevant without containing the answer. That's where
weak prompts break — the model recognizes the topic and explains from training. The France
case only proves the retriever works.

In [75]:
print(rag_chain.invoke("Who created Python?"))

Python was developed by Guido Van Rossum.


In [76]:
print(rag_chain.invoke("What is the capital of France?"))

No relevant docs were retrieved using the relevance score threshold 0.5


Sorry, this information is not available in given pdf


In [77]:
print(rag_chain.invoke("What is a Python decorator?"))   

No relevant docs were retrieved using the relevance score threshold 0.5


Sorry, this information is not available in given pdf


## 12. Validate the threshold

A threshold strict enough to reject unrelated questions can also reject *real* ones —
a **false refusal**, where the answer is in the PDF but the app says it isn't. This checks
both directions, using retrieval only (no LLM calls).

Result: all five in-scope questions returned 3 docs, both out-of-scope returned 0. Clean
separation, no tuning required.


In [79]:
in_pdf_q = [
    "What is a list in Python?",
    "What are the features of Python?",
    "What is PVM?",
    "What are Python data types?",
    "How do you write a for loop in Python?",
]

out_pdf_q = [
    "What is the capital of France?",
    "How do I bake bread?",
]

print("in_pdf:")
for q in in_pdf_q:
    print(f"  {len(retriever3.invoke(q))} docs | {q}")

print("\nout_pdf:")
for q in out_pdf_q:
    print(f"  {len(retriever3.invoke(q))} docs | {q}")

in_pdf:
  3 docs | What is a list in Python?
  3 docs | What are the features of Python?
  3 docs | What is PVM?
  3 docs | What are Python data types?
  3 docs | How do you write a for loop in Python?

out_pdf:


No relevant docs were retrieved using the relevance score threshold 0.5
No relevant docs were retrieved using the relevance score threshold 0.5


  0 docs | What is the capital of France?
  0 docs | How do I bake bread?


## Summary

| Decision | Choice | Why |
|---|---|---|
| Loader | `PyPDFLoader` | Per-page Documents with metadata |
| Chunk size | 1400 / 150 overlap | 97 chunks — under the 100/min embedding cap |
| Embeddings | `gemini-embedding-001` | Must match between index and queries |
| Vector store | FAISS (saved to disk) | Local, fast, avoids re-embedding |
| Retriever | Score threshold 0.5 | Returns 0 docs when nothing is relevant |
| Grounding | Prompt rules + empty context | Two independent layers |

**Key lesson:** debug bottom-up. Retrieval is free and testable; generation is quota-capped.
Every stage through retrieval was verified before a single LLM call was spent.
